# Model Results & Visualization Analysis

Comprehensive analysis of trained model results and feature importance.
**Purpose**: Visualize and interpret model predictions and feature contributions.
**Run Order**: AFTER 02_model_training.ipynb, 03_advanced_2stage_pipeline.ipynb, OR 05_grade_prediction_models.ipynb

## Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support

# Configuration
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 7)
sns.set_style('whitegrid')
sns.set_palette('husl')

# Paths
data_path = Path('../data')
results_path = Path('../results')

print("✓ Setup complete")

## Load Cleaned Data for Reference

In [ ]:
# Load cleaned data for context
df = pd.read_csv(data_path / 'cleaned_yrbs_data.csv')

print(f"Dataset shape: {df.shape}")

# Grade mapping
grade_map = {0: 'F', 1: 'D', 2: 'C', 3: 'B', 4: 'A'}
grade_counts = df['grade'].value_counts().sort_index()
grade_labels = [grade_map[int(g)] for g in grade_counts.index]

print(f"\nTarget Distribution:")
for grade, count in grade_counts.items():
    pct = count / grade_counts.sum() * 100
    print(f"  {grade_map[grade]}: {count:,} ({pct:.1f}%)")

## 2-Stage Pipeline Results

In [ ]:
# Load model results if available
try:
    results_2stage = pd.read_csv(results_path / '2-stages_results.csv')
    print(f"✓ 2-Stage pipeline results loaded: {results_2stage.shape[0]} samples")
    print(f"\nResults Preview:")
    print(results_2stage.head())
    
    # Confusion matrix for 2-stage
    cm = confusion_matrix(results_2stage['Actual'], results_2stage['Predicted'])
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=['F', 'D', 'C', 'B', 'A']).plot(ax=ax, cmap='Blues')
    ax.set_title('2-Stage Pipeline - Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Accuracy by grade
    prec, rec, f1, _ = precision_recall_fscore_support(results_2stage['Actual'], results_2stage['Predicted'])
    
    metrics_df = pd.DataFrame({
        'Grade': ['F', 'D', 'C', 'B', 'A'],
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print("\nPer-Class Metrics (2-Stage Pipeline):")
    print(metrics_df.round(3))
    
    # Overall accuracy
    overall_accuracy = (results_2stage['Actual'] == results_2stage['Predicted']).sum() / len(results_2stage)
    print(f"\nOverall Accuracy: {overall_accuracy:.3f}")
except FileNotFoundError:
    print("⚠ 2-Stage results file not found - skipping this section")
except Exception as e:
    print(f"⚠ Error loading 2-Stage results: {e}")

## Feature Importance - Stage 1 (F-Detection)

In [ ]:
try:
    feat_s1 = pd.read_csv(results_path / 'feature_importance_stage1_f_detector.csv')
    
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feat_s1.head(15), palette='Reds_r', ax=ax)
    ax.set_title('Stage 1: Top 15 Features for F-Detection (Binary Classifier)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nTop 15 Features - Stage 1 (F-Detection):")
    print(feat_s1.head(15).to_string(index=False))
    
except FileNotFoundError:
    print("⚠ Stage 1 feature importance file not found")
except Exception as e:
    print(f"⚠ Error loading Stage 1 features: {e}")

## Feature Importance - Stage 2 (Grade Classification)

In [ ]:
try:
    feat_s2 = pd.read_csv(results_path / 'feature_importance_stage2_grades.csv')
    
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feat_s2.head(15), palette='viridis', ax=ax)
    ax.set_title('Stage 2: Top 15 Features for Grade Classification (D/C/B/A)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nTop 15 Features - Stage 2 (Grade Classification):")
    print(feat_s2.head(15).to_string(index=False))
    
except FileNotFoundError:
    print("⚠ Stage 2 feature importance file not found")
except Exception as e:
    print(f"⚠ Error loading Stage 2 features: {e}")

## Feature Importance - XGBoost (5-Class)

In [ ]:
try:
    feat_grouped = pd.read_csv(results_path / 'feature_importance_grouped.csv')
    
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feat_grouped.head(20), palette='viridis', ax=ax)
    ax.set_title('XGBoost (5-Class) - Top 20 Feature Importance', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nTop 20 Features - XGBoost (5-Class):")
    print(feat_grouped.head(20).to_string(index=False))
    
except FileNotFoundError:
    print("⚠ Grouped feature importance file not found")
except Exception as e:
    print(f"⚠ Error loading grouped features: {e}")

## Feature Importance - All Features

## Model Comparison (if available)

In [ ]:
try:
    model_comparison = pd.read_csv(results_path / 'model_comparison_results.csv')
    
    print("\nModel Performance Comparison:")
    print(model_comparison.to_string(index=False))
    
    # Visualization
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    if all(m in model_comparison.columns for m in metrics):
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        for idx, metric in enumerate(metrics):
            ax = axes[idx]
            model_comparison_sorted = model_comparison.sort_values(metric, ascending=True)
            ax.barh(model_comparison_sorted['Model'], model_comparison_sorted[metric])
            ax.set_title(f'{metric} Comparison', fontweight='bold', fontsize=12)
            ax.set_xlabel(metric)
            ax.grid(axis='x', alpha=0.3)
        
        plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
except FileNotFoundError:
    print("⚠ Model comparison file not found")
except Exception as e:
    print(f"⚠ Error loading model comparison: {e}")

## Grade Predictions (if available)

In [ ]:
try:
    predictions = pd.read_csv(results_path / 'grade_predictions.csv')
    
    print(f"Total Predictions: {len(predictions)}")
    print(f"\nSample Predictions:")
    print(predictions.head(10))
    
    # Prediction accuracy
    if 'Actual' in predictions.columns and 'Predicted' in predictions.columns:
        accuracy = (predictions['Actual'] == predictions['Predicted']).sum() / len(predictions)
        print(f"\nPrediction Accuracy: {accuracy:.3f}")
    
except FileNotFoundError:
    print("⚠ Grade predictions file not found")
except Exception as e:
    print(f"⚠ Error loading predictions: {e}")

## Key Insights from Model Results

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS FROM MODEL ANALYSIS")
print("="*70)

# Load all available results
insights = {}

try:
    if 'feat_grouped' in dir():
        print("\n1. MOST IMPORTANT FEATURES (XGBoost 5-Class):")
        for idx, row in feat_grouped.head(10).iterrows():
            print(f"   {idx+1:2d}. {row['Feature']:30s}: {row['Importance']:.4f}")
except:
    pass

try:
    if 'results_2stage' in dir():
        print("\n2. 2-STAGE PIPELINE PERFORMANCE:")
        accuracy = (results_2stage['Actual'] == results_2stage['Predicted']).sum() / len(results_2stage)
        print(f"   • Overall Accuracy: {accuracy:.3f}")
        
        for grade_val in sorted(results_2stage['Actual'].unique()):
            grade_label = grade_map[int(grade_val)]
            grade_mask = results_2stage['Actual'] == grade_val
            grade_accuracy = (results_2stage.loc[grade_mask, 'Actual'] == results_2stage.loc[grade_mask, 'Predicted']).sum() / grade_mask.sum()
            print(f"   • Grade {grade_label} Accuracy: {grade_accuracy:.3f}")
except:
    pass

print("\n3. FEATURE IMPORTANCE SUMMARY:")
print("   → Use these insights to:")
print("     • Identify key factors affecting student grades")
print("     • Target interventions on high-impact factors")
print("     • Understand alcohol's role in academic performance")

print("\n" + "="*70)
print("✓ Results Analysis Complete")
print("="*70)